In [100]:
import pygsheets # use 'pip install pygsheets', not maintained with conda
import pandas
import datetime
import geopandas
import shapely
import pathlib
import pyogrio
import json
import numpy
import re

# terminals

## clean up terminals

# pipelines

### comment out appropriate final lines to only publish oil or gas

In [101]:
gas_fuel_options = ['Gas']
gas_hydrogen_fuel_options = ['Gas',
                             'Hydrogen'
                            ]
ngl_fuel_options = ['NGL', 
                    'NGL, oil products', 
                    'Oil products (only)',
                    'Oil, NGL', 
                    'Oil, NGL, naphtha',
                    'Naphtha (only)',
                    'Naphtha, oil products'
                   ]
oil_fuel_options = ['Oil', 
                    #'Oil, NGL', 
                    #'Oil, NGL, naphtha'
                   ]

In [102]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
#spreadsheet = gc.open_by_key('1VnhD3K8bUn-CwTGUt-XkF42jcWAaZwasKIryEO7V1j4') # may 2024 oil release
spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek') # current version

# SPECIFY HERE WHETHER YOU WANT GAS, GAS+HYDROGEN, OR OIL+NGL
#type_to_save='Oil-NGL'; pipes_df_orig = spreadsheet.worksheet('title','Oil/NGL pipelines').get_as_df(start='A3'); github_folder = 'liquid-pipelines'; which_tracker='GOIT'
#type_to_save='Gas'; pipes_df_orig = spreadsheet.worksheet('title','Gas pipelines').get_as_df(start='A3'); github_folder = 'gas-pipelines'; which_tracker='GGIT'
type_to_save='Gas-Hydrogen'; which_tracker='GGIT'; pipes_df_orig = spreadsheet.worksheet('title','Gas pipelines').get_as_df(start='A3'); github_folder = 'gas-pipelines'; 

pipes_acronyms_df = spreadsheet.worksheet('title','Acronyms').get_as_df()

In [103]:
if type_to_save=='Gas':
    pipes_dict_df = spreadsheet.worksheet('title','Data dictionary - Gas pipelines').get_as_df()
    pipes_copyright_df = spreadsheet.worksheet('title','Copyright - GGIT').get_as_df()
    pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel.isin(gas_fuel_options)]
    
elif type_to_save=='Gas-Hydrogen':
    pipes_dict_df = spreadsheet.worksheet('title','Data dictionary - Gas pipelines').get_as_df()
    pipes_copyright_df = spreadsheet.worksheet('title','Copyright - GGIT').get_as_df()
    pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel.isin(gas_hydrogen_fuel_options)]
    
elif type_to_save=='Oil-NGL':
    pipes_dict_df = spreadsheet.worksheet('title','Data dictionary - Oil/NGL pipelines').get_as_df()
    pipes_copyright_df = spreadsheet.worksheet('title','Copyright - GOIT').get_as_df()
    pipes_copyright_df = pandas.DataFrame(pipes_copyright_df.iloc[:,0])
    pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel.isin(oil_fuel_options+ngl_fuel_options)]

    # change all oil/NGL fuels to JUST SAY Oil or NGL (rather than how we track, with different options)
    pipes_df_orig.loc[pipes_df_orig.Fuel.isin(oil_fuel_options),'Fuel'] = 'Oil'
    pipes_df_orig.loc[pipes_df_orig.Fuel.isin(ngl_fuel_options),'Fuel'] = 'NGL'

    # remove gathering, distribution lines
    #pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.PipelineType.isin(['','transmission'])]

/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/pygsheets/worksheet.py:1554: UserWarning: At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.
  warnings.warn('At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.')


## clean up pipelines

In [104]:
# pipelines cleanup
# remove all N/A status, all empty rows (gauged by those without a Wiki page)
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Status!='N/A']
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.PipelineName!='']
#pipes_df_orig = pipes_df_orig[pipes_df_orig['Wiki']!='']

## ONLY save pipelines that have something in RouteAccuracy
this is the one column that decides whether we include it

In [105]:
# pipelines cleanup
# if the RouteAccuracy is blank, that means we don't want it included in the database
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.RouteAccuracy!='']
all_pipes_df = pipes_df_orig.copy()

# get github routes

In [106]:
# geojson_path = '/Users/baird/Dropbox/_git_ALL/_github-repos-gem/'+\
# 'GOIT-GGIT-pipeline-routes/data/individual-routes/'+github_folder+'/'
geojson_path = '/Users/baird/Google Drive/Shared drives/GEM Shared Drive/Programs/'+\
'Oil & Gas Program/Global Oil and Gas Infrastructure Trackers (GOIT, GGIT)/'+\
'GitHub repositories/GOIT-GGIT-pipeline-routes/data/individual-routes/'+github_folder+'/'

geojson_route_files = list(
    pathlib.Path(
        geojson_path
    ).rglob('?????.geojson')
)

In [107]:
geojson_projectids = [str(i).split('/')[-1][:5] for i in geojson_route_files]

In [108]:
set(geojson_projectids)-set(all_pipes_df.ProjectID.tolist())
# P2041 is an oil pipeline
# P3162, P3656, P3672 are all N/A but exist as files

{'P2041', 'P3162', 'P3656', 'P3672'}

In [109]:
set(all_pipes_df.ProjectID.tolist())-set(geojson_projectids)

{'P6641'}

In [110]:
missing_list =list(set(all_pipes_df.ProjectID.tolist())-set(geojson_projectids))

all_pipes_df.loc[all_pipes_df.ProjectID.isin(missing_list)]['Researcher'].unique()

array(['HH'], dtype=object)

### looking at which researcher has missing pipelines

In [111]:
all_pipes_df.loc[(all_pipes_df.ProjectID.isin(missing_list))&
                (all_pipes_df.Researcher=='HH')][['ProjectID','Fuel']].to_excel('missing-harvey.xlsx')#.tolist()]

## below are the pipelines that don't fall within the status

these may have been deleted from the spreadsheet but not from the database, or they have a N/A status, or something

In [112]:
misfits_list = list(set(geojson_projectids)-set(all_pipes_df.ProjectID.tolist()))

In [113]:
sorted(misfits_list)

['P2041', 'P3162', 'P3656', 'P3672']

In [114]:
simple_decimals_re = re.compile(r"\d*\.\d+")

def mround(match):
    return "{:.6f}".format(float(match.group()))

#pipes_df_orig['geometry'] = numpy.nan
for idx,pid in enumerate(all_pipes_df.ProjectID.tolist()):
    print(pid)
    try:
        file = geojson_path + pid + '.geojson'
        file_gdf = geopandas.read_file(file)
        file_geometry_dissolved = file_gdf.dissolve().geometry
        #file_geometry_dissolved = re.sub(simple_decimals_re, mround, file_geometry_dissolved.values[0].wkt)
        #all_pipes_df.loc[all_pipes_df.ProjectID==pid,'geometry'] = shapely.from_wkt(file_geometry_dissolved)
        all_pipes_df.loc[all_pipes_df.ProjectID==pid,'geometry'] = shapely.from_wkt(file_geometry_dissolved.values[0].wkt)
    except:
        print(pid,"geojson file either doesn't exist or has bad geometry")

P0061
P0143
P0144
P0145
P0146
P0147
P0149
P0150
P0151
P0152
P0153
P0154
P0155
P0156
P0157
P0158
P0159
P0160
P0161
P0162
P0163
P0164
P0165
P0166
P0167
P0168
P0168 geojson file either doesn't exist or has bad geometry
P0169
P0170
P0171
P0172
P0173
P0174
P0175
P0176
P0177
P0178
P0179
P0180
P0182
P0184
P0185
P0186
P0187
P0188
P0189
P0190
P0191
P0192
P0193
P0194
P0195
P0196
P0197


/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/shapely/set_operations.py:406: RuntimeWarning: invalid value encountered in create_collection
  collections = lib.create_collection(geometries, GeometryType.GEOMETRYCOLLECTION)


P0198
P0199
P0200
P0201
P0202
P0203
P0204
P0205
P0207
P0208
P0209
P0210
P0211
P0212
P0213
P0214
P0215
P0216
P0217
P0219
P0220
P0221
P0222
P0223
P0224
P0225
P0226
P0230
P0231
P0232
P0233
P0234
P0235
P0236
P0237
P0238
P0239
P0240
P0241
P0242
P0243
P0244
P0245
P0246
P0248
P0249
P0250
P0251
P0252
P0253
P0254
P0255
P0256
P0257
P0258
P0259
P0260
P0261
P0262
P0263
P0264
P0265
P0266
P0267
P0268
P0269
P0270
P0271
P0272
P0273
P0274
P0275
P0276
P0277
P0278
P0279
P0280
P0281
P0282
P0283
P0284
P0285
P0286
P0287
P0288
P0289
P0290
P0291
P0292
P0293
P0294
P0296
P0297
P0298
P0300
P0302
P0303
P0304
P0305
P0306
P0307
P0308
P0309
P0310
P0311
P0312
P0313
P0314
P0315
P0316
P0317
P0318
P0319
P0320
P0321
P0322
P0323
P0325
P0327
P0328
P0329
P0330
P0331
P0332
P0339
P0355
P0359
P0367
P0369
P0370
P0371
P0373
P0374
P0375
P0376
P0377
P0378
P0380
P0382
P0383
P0386
P0387
P0392
P0404
P0405
P0407
P0409
P0411
P0412
P0413
P0415
P0416
P0417
P0418
P0419
P0420
P0422
P0423
P0424
P0425
P0426
P0427
P0429
P0430
P0431
P0433
P043

/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/shapely/set_operations.py:406: RuntimeWarning: invalid value encountered in create_collection
  collections = lib.create_collection(geometries, GeometryType.GEOMETRYCOLLECTION)


P0934
P0935
P0936
P0937
P0938
P0941
P0943
P0944
P0953
P0954
P0955
P0957
P0958
P0959
P0960
P0961
P0962
P0963
P0964
P0965
P0966
P0967
P0968
P0969
P0970
P0971
P0972
P0974
P0975
P0976
P0977
P0978
P0979
P0980
P0981
P0982
P0983
P0984
P0985
P0988
P0990
P0991
P0992
P0993
P0994
P0995
P0996
P0997
P0998
P0999
P1000
P1001
P1002
P1003
P1004
P1005
P1006
P1007
P1012
P1013
P1014
P1015
P1016
P1017
P1018
P1019
P1020
P1021
P1022
P1023
P1024
P1025
P1026
P1027
P1028
P1029
P1030
P1031
P1032
P1033
P1034
P1036
P1037
P1038
P1039
P1041
P1042
P1043
P1044
P1045
P1046
P1047
P1048
P1049
P1050
P1051
P1052
P1054
P1055
P1056
P1057
P1058
P1059
P1060
P1061
P1062
P1063
P1065
P1066
P1067
P1068
P1071
P1072
P1073
P1076
P1078
P1079
P1080
P1081
P1082
P1083
P1085
P1086
P1087
P1088
P1089
P1090
P1091
P1095
P1096
P1097
P1098
P1100
P1101
P1102
P1103
P1104
P1105
P1106
P1107
P1108
P1109
P1110
P1111
P1112
P1113
P1114
P1115
P1116
P1117
P1118
P1119
P1121
P1122
P1123
P1124
P1126
P1127
P1128
P1129
P1130
P1131
P1133
P1134
P1135
P1136
P113

/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/fiona/collection.py:293: FeatureWarning: Empty field name at index 0
  self._schema = self.session.get_schema()
/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/geopandas/geodataframe.py:652: UserWarning: Empty field name at index 0
  for feature in features_lst:


P2235
P2236
P2237
P2239
P2241
P2242
P2243
P2244
P2245
P2246
P2247
P2248
P2249
P2250
P2251
P2252
P2253
P2254
P2256
P2258
P2259
P2265
P2266
P2267
P2270
P2271
P2272
P2273
P2275
P2276
P2277
P2278
P2282
P2283
P2285
P2286
P2287
P2288
P2289
P2290
P2291
P2292
P2296
P2297
P2298
P2299
P2300
P2309
P2310
P2312
P2313
P2315
P2320
P2321
P2322
P2326
P2327
P2334
P2335
P2336
P2337
P2339
P2340
P2341
P2342
P2343
P2346
P2348
P2350
P2351
P2352
P2353
P2355
P2357
P2358
P2362
P2364
P2365
P2366
P2370
P2371
P2372
P2375
P2376
P2378
P2379
P2380
P2381
P2383
P2385
P2387
P2388
P2390
P2424
P2425
P2426
P2429
P2430
P2431
P2432
P2433
P2435
P2437
P2438
P2440
P2447
P2450
P2458
P2459
P2462
P2463
P2464
P2466
P2467
P2469
P2470
P2471
P2472
P2473
P2474
P2475
P2476
P2477
P2479
P2480
P2482
P2484
P2485
P2486
P2487
P2488
P2489
P2490
P2491
P2492
P2493
P2494
P2495
P2496
P2497
P2498
P2499
P2500
P2501
P2502
P2505
P2509
P2510
P2516
P2517
P2518
P2519
P2521
P2522
P2523
P2524
P2526
P2529
P2530
P2531
P2536
P2537
P2538
P2539
P2541
P2542
P254

/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/shapely/set_operations.py:406: RuntimeWarning: invalid value encountered in create_collection
  collections = lib.create_collection(geometries, GeometryType.GEOMETRYCOLLECTION)
/Users/baird/miniconda3/envs/gem/lib/python3.11/site-packages/shapely/set_operations.py:419: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


P6707
P6708
P6709
P6710
P6711
P6712
P6713
P6714
P6715
P6716
P6717
P6718
P6719
P6720
P6721
P6722
P6723
P6724
P6725
P6726
P6727
P6728
P6729
P6730
P6731
P6732
P6733
P6734
P6735
P6736
P6737
P6738
P6739
P6740
P6741
P6742
P6743
P6744
P6745
P6746
P6747
P6748
P6749
P6750
P6751
P6752
P6753
P6754
P6755
P6756
P6757
P6758
P6759
P6760
P6761
P6762
P6763
P6764
P6765
P6766
P6767
P6768
P6769
P6770
P6771
P6772
P6773
P6774
P6775
P6777
P6778
P6779
P6780
P6781
P6782
P6783
P6784
P6785
P6786
P6787
P6788
P6789
P6790
P6791
P6792
P6793
P6794
P6795
P6796
P6797
P6798
P6799
P6800
P6801
P6802
P6803
P6804
P6805
P6806
P6807
P6808
P6809
P6810
P6811
P6812
P6813
P6814
P6815
P6816
P6817
P6818
P6819
P6820
P6821
P6822
P6823
P6824
P6825
P6826
P6827
P6828
P6829
P6830
P6831
P6832
P6833
P6834
P6835
P6836
P6837
P6838
P6839
P6840
P6841
P6842
P6843
P6844
P6845
P6846
P6847
P6849
P6850
P6851
P6848
P6852
P6853
P6854
P6855
P6856
P6857
P6858
P6859
P6860
P6861
P6862
P6863
P6864
P6865
P6866
P6867
P6868
P6869
P6870
P6871
P6872
P6873
P687

In [115]:
all_pipes_gdf = geopandas.GeoDataFrame(all_pipes_df, geometry=all_pipes_df.geometry, crs='EPSG:4326')

In [116]:
all_pipes_gdf.loc[all_pipes_gdf.geometry.isnull()].shape

(2, 137)

## subset pipeline columns to include relevant columns

Note these columns are specified in the "Data dictionary" tab of the Pipelines_Current sheet; the column IncludeWithDataRelease has a Yes if the column is included, and DataReleaseColumnOrder determines how they're arranged.

In [117]:
pipes_dict_df_include = pipes_dict_df.loc[(pipes_dict_df.IncludeWithDataRelease=='Yes')]
pipes_dict_df_include = pipes_dict_df_include.sort_values('DataReleaseColumnOrder', ascending=True)
pipes_dict_df_include = pipes_dict_df_include[['VariableName','Definition']]

In [118]:
all_pipes_gdf_subset = all_pipes_gdf[pipes_dict_df_include['VariableName'].tolist()]

In [119]:
all_pipes_gdf_subset_with_geom = all_pipes_gdf[pipes_dict_df_include['VariableName'].tolist()+['geometry']]

# subset only the pipelines with compressor stations

In [162]:
# geojson_path = '/Users/baird/Dropbox/_git_ALL/_github-repos-gem/'+\
# 'GOIT-GGIT-pipeline-routes/data/individual-routes/'+github_folder+'/'
geojson_path = '/Users/baird/Google Drive/Shared drives/GEM Shared Drive/Programs/'+\
'Oil & Gas Program/Global Oil and Gas Infrastructure Trackers (GOIT, GGIT)/'+\
'GitHub repositories/GOIT-GGIT-pipeline-routes/data/individual-routes/'+github_folder+'/'

geojson_compressor_station_files = list(
    pathlib.Path(
        geojson_path
    ).rglob('?????-compressor-stations.geojson')
)

In [163]:
geojson_compressor_station_files.__len__()

49

In [164]:
compressor_projectids = [str(i).split('/')[-1][:5] for i in geojson_compressor_station_files]
all_pipes_gdf_subset = all_pipes_gdf_subset.loc[all_pipes_gdf_subset.ProjectID.isin(compressor_projectids)]

## write to Excel file

Use ExcelFormatter from Pandas to write 4 different tabs to the same file.

In [206]:
#pandas.io.formats.excel.ExcelFormatter.header_style = None

excel_writer = pandas.ExcelWriter('DATA-FILES/GEM-'+which_tracker+'-'+type_to_save+'-Pipelines-RMI.xlsx')#, engine='xlsxwriter')

all_pipes_gdf_subset.loc[all_pipes_gdf_subset.ProjectID.isin(compressor_projectids)].to_excel(excel_writer, sheet_name=type_to_save+' Pipelines '+str(datetime.date.today())[:7], index=False)
pipes_dict_df_include.to_excel(excel_writer, sheet_name='Data dictionary', index=False)
pipes_acronyms_df.to_excel(excel_writer, sheet_name='Acronyms', index=False)
pipes_copyright_df.to_excel(excel_writer, sheet_name='Copyright', index=False)

#workbook = excel_writer.book
#worksheet = excel_writer.sheets['Terminals '+str(datetime.date.today())]
#format = workbook.add_format({'text_wrap': True})

excel_writer.close()

## save as GeoJSON file

In [207]:
str(datetime.date.today())[:10]

'2024-10-29'

In [208]:
filename = 'DATA-FILES/GEM-'+which_tracker+'-'+type_to_save+'-Pipelines-RMI.geojson'
all_pipes_gdf_subset_with_geom.loc[all_pipes_gdf_subset_with_geom.ProjectID.isin(compressor_projectids)].to_file(filename, driver='GeoJSON')
print('saved as', filename)

saved as DATA-FILES/GEM-GGIT-Gas-Hydrogen-Pipelines-RMI.geojson


# create compressor station file

In [204]:
compressor_stations_df = geopandas.GeoDataFrame()#index=compressor_projectids)

for pid in compressor_projectids:
    file = geojson_path + pid + '-compressor-stations.geojson'
    file_gdf = geopandas.read_file(file)
    file_gdf['ProjectID'] = pid
    file_gdf['PipelineName'] = all_pipes_gdf.loc[all_pipes_gdf.ProjectID==pid].PipelineName.values[0]
    #print(file_gdf['PipelineName'])
    #print(pid, file_gdf.crs)
    #print(pid)
    compressor_stations_df = geopandas.GeoDataFrame( 
       pandas.concat( [compressor_stations_df, file_gdf[['ProjectID','PipelineName','geometry']] ], ignore_index=True
                    )
    )
    

In [ ]:
filename = 'DATA-FILES/GEM-'+which_tracker+'-'+type_to_save+'-Compressor-Stations-RMI.geojson'
compressor_stations_df.to_file(filename, driver='GeoJSON')
print('saved as', filename)

In [205]:
filename = 'DATA-FILES/GEM-'+which_tracker+'-'+type_to_save+'-Compressor-Stations-RMI.geojson'
compressor_stations_df.to_file(filename, driver='GeoJSON')
print('saved as', filename)

saved as DATA-FILES/GEM-GGIT-Gas-Hydrogen-Compressor-Stations-RMI.geojson
